In [ ]:
"""
REAL WORLD WORKFLOW

1. Data Understanding
2. EDA
3. Feature Selection
4. Scaling
5. Baseline GMM
6. Cluster Assignment
7. Cluster Probabilities
8. Internal Validation
9. Hyperparameter Tuning
10. AIC/BIC Analysis
11. Covariance Type Comparison
12. PCA Visualization
13. Cluster Profiling
14. Deployment Readiness
15. Interview Questions

==================================================================
"""

# ==========================================================
# STEP 0 : IMPORTS
# ==========================================================

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score,davies_bouldin_score,calinski_harabasz_score)
from sklearn.decomposition import PCA
import joblib


In [ ]:
# ==========================================================
# STEP 1 : DATA UNDERSTANDING
# ==========================================================

wine = load_wine()
df = pd.DataFrame(wine.data,columns=wine.feature_names)

print("\nHEAD")
print(df.head())
print("\nINFO")
print(df.info())
print("\nDESCRIBE")
print(df.describe())
print("\nSHAPE")
print(df.shape)

In [ ]:
# ==========================================================
# STEP 2 : EDA
# ==========================================================

print("\nMISSING VALUES")
print(df.isnull().sum())
print("\nDUPLICATES")
print(df.duplicated().sum())
plt.figure(figsize=(12,8))
sns.heatmap(df.corr(),cmap='coolwarm')

plt.title("Correlation Matrix")
plt.show()

In [ ]:
# ==========================================================
# STEP 3 : FEATURE SELECTION
# ==========================================================

X = df.copy()

In [ ]:
# ==========================================================
# STEP 4 : SCALING
# ==========================================================

"""
GMM uses probability distributions
and covariance matrices.

Scaling is strongly recommended.
"""

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# ==========================================================
# STEP 5 : BASELINE GMM
# ==========================================================

gmm = GaussianMixture(n_components=3,covariance_type='full',max_iter=200,n_init=10,random_state=42)

"""
Important Parameters:

n_components
-------------
Number of clusters

covariance_type
---------------
full      -> each cluster has own covariance matrix
tied      -> all clusters share covariance
diag      -> diagonal covariance
spherical -> spherical covariance

n_init
-------
Number of initializations

max_iter
---------
EM iterations
"""

In [ ]:
# ==========================================================
# STEP 6 : CLUSTER ASSIGNMENT
# ==========================================================

labels = gmm.fit_predict(X_scaled)
df["cluster"] = labels
print("\nCluster Counts")
print(df["cluster"].value_counts())

In [ ]:
# ==========================================================
# STEP 7 : CLUSTER PROBABILITIES
# ==========================================================

"""
KMeans: Hard Clustering
GMM: Soft Clustering
"""

probabilities = gmm.predict_proba(X_scaled)
print("\nCluster Membership Probabilities")
print(probabilities[:5])

prob_df = pd.DataFrame(probabilities, columns=[f"cluster_{i}_prob"for i in range(probabilities.shape[1])])
print("\nProbability DataFrame")
print(prob_df.head())

In [ ]:
# ==========================================================
# STEP 8 : INTERNAL VALIDATION
# ==========================================================

sil = silhouette_score(X_scaled,labels)
db = davies_bouldin_score(X_scaled,labels)
ch = calinski_harabasz_score(X_scaled,labels)

print("\nSilhouette Score:", sil)
print("Davies Bouldin Score:", db)
print("Calinski Harabasz Score:", ch)

In [ ]:
# ==========================================================
# STEP 9 : HYPERPARAMETER TUNING
# ==========================================================

results = []

for n in range(2,11):
    model = GaussianMixture(n_components=n,covariance_type='full',random_state=42)
    lbl = model.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled,lbl)
    results.append([n,sil])
    
tuning_df = pd.DataFrame(results,columns=["Components","Silhouette"])
print("\nHyperparameter Search")
print(tuning_df)

In [ ]:
# ==========================================================
# STEP 10 : AIC / BIC ANALYSIS
# ==========================================================

"""
Most Important GMM Validation
Lower AIC -> Better
Lower BIC -> Better
"""

aic_scores = []
bic_scores = []
components = range(1,11)

for n in components:
    model = GaussianMixture(n_components=n,covariance_type='full',random_state=42)
    model.fit(X_scaled)
    aic_scores.append(model.aic(X_scaled))
    bic_scores.append(model.bic(X_scaled))

plt.figure(figsize=(10,5))
plt.plot(components,aic_scores,marker='o',label="AIC")
plt.plot(components,bic_scores,marker='o',label="BIC")
plt.xlabel("Components")
plt.ylabel("Score")
plt.title("AIC / BIC Analysis")
plt.legend()
plt.show()

best_aic = components[np.argmin(aic_scores)]
best_bic = components[np.argmin(bic_scores)]
print("\nBest AIC Components:", best_aic)
print("Best BIC Components:", best_bic)

In [ ]:
# ==========================================================
# STEP 11 : COVARIANCE COMPARISON
# ==========================================================

cov_types = ["full","tied","diag","spherical"]
comparison = []

for cov in cov_types:
    model = GaussianMixture(n_components=3,covariance_type=cov,random_state=42)
    lbl = model.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled,lbl)
    comparison.append([cov,sil])
comparison_df = pd.DataFrame(comparison,columns=["Covariance","Silhouette"])

print("\nCovariance Comparison")
print(comparison_df)

In [ ]:
# ==========================================================
# STEP 12 : PCA VISUALIZATION
# ==========================================================

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
plt.figure(figsize=(10,6))
scatter = plt.scatter(X_pca[:,0],X_pca[:,1],c=labels,cmap="viridis")

plt.colorbar(scatter)
plt.title("GMM Clusters")
plt.xlabel("PCA1")
plt.ylabel("PCA2")
plt.show()

In [ ]:
# ==========================================================
# STEP 13 : CLUSTER PROFILING
# ==========================================================

profile = df.groupby("cluster").mean()
print("\nCluster Profile")
print(profile)
print("\nCluster Counts")
print(df["cluster"].value_counts())

In [ ]:
# ==========================================================
# STEP 14 : DEPLOYMENT READINESS
# ==========================================================

joblib.dump(gmm,"gmm_model.pkl")
joblib.dump(scaler,"gmm_scaler.pkl")

print("\nArtifacts Saved")
print("gmm_model.pkl")
print("gmm_scaler.pkl")

# Example New Prediction
sample = X.iloc[[0]]
sample_scaled = scaler.transform(sample)
cluster = gmm.predict(sample_scaled)
prob = gmm.predict_proba(sample_scaled)

print("\nPredicted Cluster")
print(cluster)
print("\nPrediction Probability")
print(prob)

# ==========================================================
# STEP 15 : INTERVIEW QUESTIONS
# ==========================================================

"""
1. What is GMM?
2. Difference between GMM and KMeans?
3. What is Soft Clustering?
4. What is Hard Clustering?
5. What is a Gaussian Distribution?
6. What is a Mixture Model?
7. What is EM Algorithm?
8. Explain Expectation Step.
9. Explain Maximization Step.
10. Why GMM can model elliptical clusters?
11. Why KMeans struggles with elliptical clusters?
12. What is covariance?
13. Explain covariance_type='full'.
14. Difference between:
    full
    tied
    diag
    spherical
15. What is AIC?
16. What is BIC?
17. Why lower AIC/BIC is better?
18. How to select n_components?
19. Why scaling is recommended?
20. How do you validate GMM?
21. What are cluster probabilities?
22. Can GMM predict new samples?
23. Real-world use cases?
24. Advantages over DBSCAN?
25. Explain GMM end-to-end.
26. Time Complexity of GMM?
27. Why EM converges?
28. What is likelihood?
29. What is log-likelihood?
30. Explain GMM mathematically.
"""

# ==========================================================
# END OF FILE
# ==========================================================